In [ ]:
################################################################################
# T1 optical only
# Single instance
#
################################################################################
#
# Python imports
#
import numpy as np
import matplotlib.pyplot as plt

import random
import sys
import tempfile
from time import sleep

# Pulse blaster wrapper functions 
from spinapi import *

# Pyvisa interfaces for the SR830 lockin amplifier and Anritsu RF signal generator
import pyvisa
from Instrument_wrappers import *


################################################################################
#
# Define and allocate temporary and output arrays
#

# log.info("Starting the loop of %d iterations" % num_delay_times)

# Tdelays_arr = np.linspace(0, MaxTdelay, num_delay_times)
# Tdelays_arr = np.logspace( np.log10(5*us), np.log10(2*ms), num_delay_times)
                   
# Tstep = MaxTdelay / num_delay_times
# LIstep = Tstep / num_data_points

################################################################################
#
# Setup and initialise the instruments : 
#   PulseBlaster, SignalGenerator, LockInAmplifier
#   pb, sg, lia are handles to our wrapper classes - not used much actually
#

pb = PulseBlaster()

sg = SignalGenerator()
siggen = sg.get_siggen()

lia = LockInAmplifier()
lockin = lia.get_lockin()

In [ ]:
################################################################################
#
# Loops in which the pulse sequence parameters change go here !
#

########################################
#
# Define some pulse sequence parameters
#
# Note that the units : ms, us are defined as multipliers in SpinCorePyWrapper 
#   all times are expected to be in ns = nanoseconds
#
Tref = 6.5*ms
Tinit = 100*us
Tread = 5*us
# Tlaser = 200*us
# Tmicrowave = 3*us

Tdelay =400*us
# Tdelay = 0.5*ms

#---------------------------------------------------------------------------
#
# Program the updated pulse program 
#
# Define binary strings for each output channel 
#   Note : here we use the first four channels (read right to left) for instrument control
#          and repeat those on channels 5-8 to monitor on an oscilloscope
#
reference = 0b00010001
laser     = 0b00100010
RF_I      = 0b01000100
RF_Q      = 0b10001000
zeros     = 0b00000000

pb_start_programming(PULSE_PROGRAM)

#
# reference high
start = pb_inst_pbonly(reference + laser, Inst.CONTINUE, 0, Tinit)
pb_inst_pbonly(reference, Inst.CONTINUE, 0, Tdelay)
pb_inst_pbonly(reference + laser, Inst.CONTINUE, 0, Tread)
pb_inst_pbonly(reference, Inst.CONTINUE, 0, Tref - Tinit - Tdelay - Tread)
# pb_inst_pbonly(reference, Inst.CONTINUE, 0, Tref - Tinit)

# reference low 
pb_inst_pbonly(laser, Inst.CONTINUE, 0, Tinit)
pb_inst_pbonly(zeros, Inst.BRANCH, start, Tref - Tinit)

pb_stop_programming()

#---------------------------------------------------------------------------
#
# Trigger the PulseBlaster to run our sequence
#

pb.trigger_pulse_program()  # reset and start playing pulse sequence
# pb_reset()   # reset everything; this must be called before pb_start
# pb_start()   # start the sequence running

sleep(10.0)   # wait a moment to ensure the pulse program is running

lockin.write('APHS')

sleep(10.0)   # wait a moment to ensure the pulse program is running

#---------------------------------------------------------------------------
# Python code to do data acquisition while the current pulse sequence runs
#

#  Just sit here and run this one program to set parameters on the lockin amplifier

# print("Pressing a key will stop pulsing\n");
# input("Please press a key to stop.")

# pb_stop()    # stop running the pulse sequence (leave communication open)

# pb_close()   # close communication with the PulseBlaster

In [ ]:
pb_stop()
# pb_close()

In [ ]:
lockin.write('APHS')

In [ ]:
lockin.query('OFLT?')

In [ ]:
################################################################################
# T1 optical only
# Single loop code
#
################################################################################

################################################################################
#
# Loops in which the pulse sequence parameters change go here !
#

########################################
#
# Define some pulse sequence parameters
#
# Note that the units : ms, us are defined as multipliers in SpinCorePyWrapper 
#   all times are expected to be in ns = nanoseconds
#
Tref = 6.5*ms
Tinit = 100*us
Tread = 5*us
# Tlaser = 200*us
# Tmicrowave = 3*us

Tdelay_min = 200*us
Tdelay_step = 200*us
# Tdelay_max = 10*ms
num_delay_times = 6
    
lockin_settle_time = 10   # units are seconds, used by Python sleep() function

# initialize output array(s)
Tdelay_array = np.zeros(num_delay_times)
vlockin_array = np.zeros(num_delay_times)


# loop over something...
Tdelay = Tdelay_min
for n in range(num_delay_times):

    #---------------------------------------------------------------------------
    #
    # Program the updated pulse program 
    #
    # Define binary strings for each output channel 
    #   Note : here we use the first four channels (read right to left) for instrument control
    #          and repeat those on channels 5-8 to monitor on an oscilloscope
    #
    reference = 0b00010001
    laser     = 0b00100010
    RF_I      = 0b01000100
    RF_Q      = 0b10001000
    zeros     = 0b00000000

    pb_start_programming(PULSE_PROGRAM)

    #
    # reference high
    start = pb_inst_pbonly(reference + laser, Inst.CONTINUE, 0, Tinit)
    pb_inst_pbonly(reference, Inst.CONTINUE, 0, Tdelay)
    pb_inst_pbonly(reference + laser, Inst.CONTINUE, 0, Tread)
    pb_inst_pbonly(reference, Inst.CONTINUE, 0, Tref - Tinit - Tdelay - Tread)
    # pb_inst_pbonly(reference, Inst.CONTINUE, 0, Tref - Tinit)

    # reference low 
    pb_inst_pbonly(laser, Inst.CONTINUE, 0, Tinit)
    pb_inst_pbonly(zeros, Inst.BRANCH, start, Tref - Tinit)

    pb_stop_programming()
    

    #---------------------------------------------------------------------------
    #
    # Trigger the PulseBlaster to run our sequence
    #

    pb.trigger_pulse_program()  # reset and start playing pulse sequence
    
    # pb_reset()   # reset everything; this must be called before pb_start
    # pb_start()   # start the sequence running

    sleep(lockin_settle_time)   # wait a moment to ensure the pulse program is running
                #   and then autophase it

    lockin.write('APHS')

    #---------------------------------------------------------------------------
    # Python code to do data acquisition while the current pulse sequence runs
    #

    sleep(lockin_settle_time)
    
    # acquire lockin signal [V] and convert to [pA]
    R = float(lockin.query('OUTP? 3')) * 1e12

    Tdelay_array[n] = Tdelay
    vlockin_array[n] = R

    print("Tdelay = " + f'{Tdelay/1e6:6.3f}' + "    Vlockin=" + f'{R:.4f}' )
    
    # pb_stop()    # stop running the pulse sequence (leave communication open)

    # update variables for this iteration    
    Tdelay += Tdelay_step

#
# end loop

# pb_close()   # close communication with the PulseBlaster

################################################################################
#
# Process and plot the output
#
plt.plot(Tdelay_array, vlockin_array)


In [ ]:
################################################################################
# T1 optical only
# Double loop code
#
################################################################################
#
# Python imports
#
import numpy as np
import matplotlib.pyplot as plt

import random
import sys
import tempfile
from time import sleep

# Pulse blaster wrapper functions 
from spinapi import *

# Pyvisa interfaces for the SR830 lockin amplifier and Anritsu RF signal generator
import pyvisa
from Instrument_wrappers import *


################################################################################
#
# Define and allocate temporary and output arrays
#

# log.info("Starting the loop of %d iterations" % num_delay_times)

# Tdelays_arr = np.linspace(0, MaxTdelay, num_delay_times)
# Tdelays_arr = np.logspace( np.log10(5*us), np.log10(2*ms), num_delay_times)
                   
# Tstep = MaxTdelay / num_delay_times
# LIstep = Tstep / num_data_points

################################################################################
#
# Setup and initialise the instruments : 
#   PulseBlaster, SignalGenerator, LockInAmplifier
#   pb, sg, lia are handles to our wrapper classes - not used much actually
#

pb = PulseBlaster()

sg = SignalGenerator()
siggen = sg.get_siggen()

lia = LockInAmplifier()
lockin = lia.get_lockin()

################################################################################
#
# Loops in which the pulse sequence parameters change go here !
#

########################################
#
# Define some pulse sequence parameters
#
# Note that the units : ms, us are defined as multipliers in SpinCorePyWrapper 
#   all times are expected to be in ns = nanoseconds
#
Tref = 15*ms
Tlaser = 5*us
# Tmicrowave = 3*us

Tlaser_min = 1*us
Tlaser_max = 30*us
num_tlaser_times = 6

Tdelay_min = 5*us
Tdelay_max = 4*ms
num_delay_times = 20
    
lockin_settle_time = 2   # units are seconds, used by Python sleep() function


Tlaser_array = np.geomspace(Tlaser_min, Tlaser_max, num_tlaser_times)

Tdelay_array = np.linspace(Tdelay_min, Tdelay_max, num_delay_times)

# initialize output array(s)
vlockin_array = np.zeros((num_tlaser_times,num_delay_times))


# loop over something else...
for m in range(num_tlaser_times):

    # update variables for this iteration    
    Tlaser = Tlaser_array[m]

    print("Next loop : Tlaser = " + f'{Tlaser/1e6:6.4f}')

    # loop over something...
    for n in range(num_delay_times):

        # update variables for this iteration    
        Tdelay = Tdelay_array[n]

        # # number of times sequences are repeated in each Tref
        # Nrepeats = num_pulses
        # if( Tref / Nrepeats < 2 * Tlaser ):
        #     print("Error in pulse sequence, Treference is too short")
        #     input("Please press a key to continue.")
        #     exit(-1)


        #---------------------------------------------------------------------------
        #
        # Program the updated pulse program 
        #
        # Define binary strings for each output channel 
        #   Note : here we use the first four channels (read right to left) for instrument control
        #          and repeat those on channels 5-8 to monitor on an oscilloscope
        #
        reference = 0b00010001
        laser     = 0b00100010
        RF_I      = 0b01000100
        RF_Q      = 0b10001000
        zeros     = 0b00000000

        pb_start_programming(PULSE_PROGRAM)


        #
        # reference high
        start = pb_inst_pbonly(reference + laser, Inst.CONTINUE, 0, Tlaser)
        pb_inst_pbonly(reference, Inst.CONTINUE, 0, Tref - Tlaser)
        # reference low 
        pb_inst_pbonly(laser, Inst.CONTINUE, 0, Tlaser)
        pb_inst_pbonly(zeros, Inst.CONTINUE, 0, Tdelay)
        pb_inst_pbonly(laser, Inst.CONTINUE, 0, Tlaser)
        pb_inst_pbonly(zeros, Inst.BRANCH, start, Tref - 2*Tlaser - Tdelay)


        pb_stop_programming()


        #---------------------------------------------------------------------------
        #
        # Trigger the PulseBlaster to run our sequence
        #

        pb.trigger_pulse_program()  # reset and start playing pulse sequence

        # pb_reset()   # reset everything; this must be called before pb_start
        # pb_start()   # start the sequence running

        sleep(0.1)   # wait a moment to ensure the pulse program is running


        #---------------------------------------------------------------------------
        # Python code to do data acquisition while the current pulse sequence runs
        #

        sleep(lockin_settle_time)

        # acquire lockin signal [V] and convert to [mV]
        R = float(lockin.query('OUTP? 3')) * 1000

        vlockin_array[m,n] = R

        print("Tdelay = " + f'{Tdelay/1e6:6.3f}' + "    Vlockin=" + f'{R:.6f}' )

        pb_stop()    # stop running the pulse sequence (leave communication open)

#
# end loop

# pb_close()   # close communication with the PulseBlaster

################################################################################
#
# Process and plot the output
#
# plt.plot(Tdelay_array, vlockin_array)
